# Dashboard Truc Quan Hoa & Thu Nghiem Du Bao Top 10 Co Phieu

Dashboard tuong tac ho tro:
1. Chon ma co phieu trong Top 10 danh muc VN100 (VCB, VIC, VHM, BID, CTG, GAS, HPG, VNM, FPT, VPB).
2. Hien thi the chi so KPI: Gia hien tai, Ty suat du bao ngay mai, Tin hieu giao dich (BUY / SELL / HOLD) va Hit Rate.
3. Bieu do tuong tac (Plotly): So sanh gia thuc te va gia du bao tu mo hinh Deep Learning (LSTM).
4. Bang doi chieu Benchmark 4 mo hinh (Naive, Linear Regression, XGBoost, LSTM) cho rieng ma duoc chon.
5. Mo phong chien luoc giao dich (Backtest Cumulative Returns) so voi chien luoc Buy & Hold.

## 1. Khai bao thu vien va cau hinh he thong

In [7]:
import os
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_squared_error, mean_absolute_error
import tensorflow as tf

# Thu vien truc quan hoa tuong tac
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Duong dan thu muc du an
BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
INTERIM_DATA_DIR = os.path.join(BASE_DIR, 'data', 'interim')
BASELINES_DIR = os.path.join(BASE_DIR, 'models', 'baselines')
DL_MODELS_DIR = os.path.join(BASE_DIR, 'models', 'deep_learning', 'top10')

TOP_10_TICKERS = [
    'VCB.VN', 'VIC.VN', 'VHM.VN', 'BID.VN', 'CTG.VN',
    'GAS.VN', 'HPG.VN', 'VNM.VN', 'FPT.VN', 'VPB.VN'
]

print("He thong da san sang voi danh muc Top 10:", TOP_10_TICKERS)

He thong da san sang voi danh muc Top 10: ['VCB.VN', 'VIC.VN', 'VHM.VN', 'BID.VN', 'CTG.VN', 'GAS.VN', 'HPG.VN', 'VNM.VN', 'FPT.VN', 'VPB.VN']


## 2. Core Engine: Nap du lieu, Chay mo hinh va Tinh toan tin hieu

In [8]:
def load_and_predict(ticker, lookback=60, threshold=0.005):
    data_path = os.path.join(INTERIM_DATA_DIR, f"{ticker}_features.csv")
    if not os.path.exists(data_path):
        raise FileNotFoundError(f"Khong tim thay tap tin {data_path}")
        
    df = pd.read_csv(data_path)
    df['Target_Return'] = df['Log_Return'].shift(-1)
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)
    df.reset_index(drop=True, inplace=True)
    
    exclude_cols = ['Date', 'Target_Return']
    feature_cols = [c for c in df.columns if c not in exclude_cols]
    
    X = df[feature_cols]
    y = df['Target_Return']
    
    test_size = int(len(df) * 0.20)
    df_test = df.iloc[-test_size:].copy().reset_index(drop=True)
    X_test = X.iloc[-test_size:].copy().reset_index(drop=True)
    y_test = y.iloc[-test_size:].copy().reset_index(drop=True)
    
    # 1. Nap Model LSTM & Scalers
    m_path = os.path.join(DL_MODELS_DIR, f"{ticker}_lstm_model.keras")
    fs_path = os.path.join(DL_MODELS_DIR, f"{ticker}_feature_scaler.pkl")
    ts_path = os.path.join(DL_MODELS_DIR, f"{ticker}_target_scaler.pkl")
    
    if not os.path.exists(m_path):
        m_path = os.path.join(DL_MODELS_DIR, "best_lstm_model.keras")
        fs_path = os.path.join(DL_MODELS_DIR, "feature_scaler.pkl")
        ts_path = os.path.join(DL_MODELS_DIR, "target_scaler.pkl")
        
    lstm_model = tf.keras.models.load_model(m_path)
    feature_scaler = joblib.load(fs_path)
    target_scaler = joblib.load(ts_path)
    
    # 2. Nap Baselines
    lr_path = os.path.join(BASELINES_DIR, f"{ticker}_lr_baseline.pkl")
    xgb_path = os.path.join(BASELINES_DIR, f"{ticker}_xgb_baseline.pkl")
    lr_model = joblib.load(lr_path) if os.path.exists(lr_path) else None
    xgb_model = joblib.load(xgb_path) if os.path.exists(xgb_path) else None
    
    # 3. Chuan bi Sequence cho LSTM
    X_test_scaled = feature_scaler.transform(X_test.values)
    valid_indices = list(range(lookback, len(X_test_scaled)))
    X_seq = np.array([X_test_scaled[i-lookback:i] for i in valid_indices])
    
    y_pred_lstm_scaled = lstm_model.predict(X_seq, verbose=0)
    y_pred_lstm = target_scaler.inverse_transform(y_pred_lstm_scaled).flatten()
    
    # Subframe cho tap valid lookback
    eval_df = df_test.iloc[valid_indices].copy().reset_index(drop=True)
    eval_df['Pred_Return_LSTM'] = y_pred_lstm
    eval_df['True_Return'] = y_test.iloc[valid_indices].values
    
    # Du doan gia dong cua ngay mai dua tren Predicted Return
    # P_{t+1} = P_t * exp(r_{t+1})
    eval_df['Predicted_Close'] = eval_df['Close'] * np.exp(eval_df['Pred_Return_LSTM'])
    
    # Tin hieu giao dich (Trading Signal)
    def get_signal(ret):
        if ret > threshold:
            return 'BUY'
        elif ret < -threshold:
            return 'SELL'
        return 'HOLD'
        
    eval_df['Signal'] = eval_df['Pred_Return_LSTM'].apply(get_signal)
    
    # 4. Tinh Benchmark Metrics
    y_true = eval_df['True_Return'].values
    metrics_list = []
    
    # Naive
    y_pred_naive = eval_df['Log_Return'].values
    # LR & XGB
    y_pred_lr = lr_model.predict(X_test.iloc[valid_indices]) if lr_model else np.zeros(len(valid_indices))
    y_pred_xgb = xgb_model.predict(X_test.iloc[valid_indices]) if xgb_model else np.zeros(len(valid_indices))
    
    for name, p in [('Naive Baseline', y_pred_naive), ('Linear Regression', y_pred_lr), ('XGBoost', y_pred_xgb), ('Deep Learning (LSTM)', y_pred_lstm)]:
        rmse = np.sqrt(mean_squared_error(y_true, p))
        mae = mean_absolute_error(y_true, p)
        t_dir = np.sign(y_true)
        p_dir = np.sign(p)
        mask = (t_dir != 0) & (p_dir != 0)
        hit_rate = (t_dir[mask] == p_dir[mask]).sum() / np.sum(mask) * 100 if np.sum(mask) > 0 else 50.0
        metrics_list.append({'Model': name, 'Hit Rate (%)': hit_rate, 'RMSE': rmse, 'MAE': mae})
        
    bench_res = pd.DataFrame(metrics_list)
    
    # 5. Mo phong Backtest chien luoc AI vs Buy & Hold
    # Position: BUY -> 1 (giu vi the), SELL -> 0 (dung ngoai/tien mat), HOLD -> giu vi the truoc
    positions = []
    curr_pos = 0
    for sig in eval_df['Signal']:
        if sig == 'BUY':
            curr_pos = 1
        elif sig == 'SELL':
            curr_pos = 0
        positions.append(curr_pos)
        
    eval_df['Position'] = positions
    eval_df['Strategy_Return'] = eval_df['Position'].shift(1).fillna(0) * eval_df['True_Return']
    eval_df['Cum_Strategy'] = (1 + eval_df['Strategy_Return']).cumprod() - 1
    eval_df['Cum_BuyHold'] = (1 + eval_df['True_Return']).cumprod() - 1
    
    return eval_df, bench_res

print("Ham load_and_predict da khoi tao thanh cong.")

Ham load_and_predict da khoi tao thanh cong.


## 3. Giao dien Dashboard tuong tac (Interactive Widget Dashboard)

In [9]:
ticker_dropdown = widgets.Dropdown(
    options=TOP_10_TICKERS,
    value='VCB.VN',
    description='Co phieu:',
    style={'description_width': 'initial'}
)

threshold_slider = widgets.FloatSlider(
    value=0.005,
    min=0.001,
    max=0.02,
    step=0.001,
    description='Nguong Signal (+/- %):',
    readout_format='.3f',
    style={'description_width': 'initial'}
)

out = widgets.Output()

def render_dashboard(ticker, threshold):
    with out:
        clear_output(wait=True)
        eval_df, bench_res = load_and_predict(ticker, lookback=60, threshold=threshold)
        
        latest_row = eval_df.iloc[-1]
        latest_close = latest_row['Close']
        pred_ret = latest_row['Pred_Return_LSTM'] * 100
        pred_close = latest_row['Predicted_Close']
        latest_signal = latest_row['Signal']
        
        lstm_hit = bench_res[bench_res['Model'] == 'Deep Learning (LSTM)']['Hit Rate (%)'].values[0]
        lstm_rmse = bench_res[bench_res['Model'] == 'Deep Learning (LSTM)']['RMSE'].values[0]
        
        # Card Style
        sig_color = '#27ae60' if latest_signal == 'BUY' else ('#e74c3c' if latest_signal == 'SELL' else '#f39c12')
        
        html_card = f"""
        <div style="display: flex; gap: 16px; margin-bottom: 20px; font-family: sans-serif;">
            <div style="flex: 1; background: #f8f9fa; border: 1px solid #e9ecef; border-radius: 8px; padding: 14px; text-align: center;">
                <div style="color: #6c757d; font-size: 13px; font-weight: bold;">GIA HIEN TAI ({ticker})</div>
                <div style="font-size: 24px; font-weight: bold; color: #2c3e50;">{latest_close:,.1f}</div>
            </div>
            <div style="flex: 1; background: #f8f9fa; border: 1px solid #e9ecef; border-radius: 8px; padding: 14px; text-align: center;">
                <div style="color: #6c757d; font-size: 13px; font-weight: bold;">GIA DU BAO NGAY MAI</div>
                <div style="font-size: 24px; font-weight: bold; color: #2980b9;">{pred_close:,.1f} <span style="font-size: 14px;">({pred_ret:+.2f}%)</span></div>
            </div>
            <div style="flex: 1; background: #f8f9fa; border: 1px solid #e9ecef; border-radius: 8px; padding: 14px; text-align: center;">
                <div style="color: #6c757d; font-size: 13px; font-weight: bold;">TIN HIEU GIAO DICH AI</div>
                <div style="font-size: 24px; font-weight: bold; color: {sig_color};">{latest_signal}</div>
            </div>
            <div style="flex: 1; background: #f8f9fa; border: 1px solid #e9ecef; border-radius: 8px; padding: 14px; text-align: center;">
                <div style="color: #6c757d; font-size: 13px; font-weight: bold;">HIT RATE TEST ({ticker})</div>
                <div style="font-size: 24px; font-weight: bold; color: #8e44ad;">{lstm_hit:.2f}%</div>
            </div>
        </div>
        """
        display(HTML(html_card))
        
        # BIEU DO 1: Gia Thuc Te vs Du Bao (Plotly)
        fig = make_subplots(
            rows=2, cols=1,
            shared_xaxes=True,
            vertical_spacing=0.08,
            subplot_titles=(
                f'Du Bao Gia Dong Cua & Tin Hieu Mua/Ban ({ticker})', 
                'Mo Phong Hieu Qua Chien Luoc AI vs Buy & Hold (Cumulative Return)'
            ),
            row_heights=[0.6, 0.4]
        )
        
        # Row 1: Price and predictions
        fig.add_trace(go.Scatter(
            x=eval_df['Date'], y=eval_df['Close'],
            mode='lines', name='Gia Thuc Te (Close)',
            line=dict(color='#2c3e50', width=2)
        ), row=1, col=1)
        
        fig.add_trace(go.Scatter(
            x=eval_df['Date'], y=eval_df['Predicted_Close'],
            mode='lines', name='LSTM Du Bao (Predicted Close)',
            line=dict(color='#e67e22', width=1.8, dash='dot')
        ), row=1, col=1)
        
        # Diem BUY / SELL
        buys = eval_df[eval_df['Signal'] == 'BUY']
        sells = eval_df[eval_df['Signal'] == 'SELL']
        
        fig.add_trace(go.Scatter(
            x=buys['Date'], y=buys['Close'],
            mode='markers', name='Tin Hieu MUA (BUY)',
            marker=dict(symbol='triangle-up', size=9, color='#27ae60')
        ), row=1, col=1)
        
        fig.add_trace(go.Scatter(
            x=sells['Date'], y=sells['Close'],
            mode='markers', name='Tin Hieu BAN (SELL)',
            marker=dict(symbol='triangle-down', size=9, color='#e74c3c')
        ), row=1, col=1)
        
        # Row 2: Backtest Returns
        fig.add_trace(go.Scatter(
            x=eval_df['Date'], y=eval_df['Cum_Strategy']*100,
            mode='lines', name='Chien Luoc Theo AI LSTM (%)',
            line=dict(color='#27ae60', width=2)
        ), row=2, col=1)
        
        fig.add_trace(go.Scatter(
            x=eval_df['Date'], y=eval_df['Cum_BuyHold']*100,
            mode='lines', name='Buy & Hold Co Phieu (%)',
            line=dict(color='#95a5a6', width=1.5, dash='dash')
        ), row=2, col=1)
        
        fig.update_layout(
            height=700,
            hovermode='x unified',
            legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
            margin=dict(l=40, r=40, t=60, b=40)
        )
        fig.update_yaxes(title_text='Gia (VND)', row=1, col=1)
        fig.update_yaxes(title_text='Loi Nhuan Tich Luy (%)', row=2, col=1)
        fig.show()
        
        # Bang doi chieu Benchmark
        display(HTML(f"<h4 style='font-family: sans-serif; color: #2c3e50; margin-top: 20px;'>Bang So Sanh Hieu Nang 4 Mo Hinh Tren Tap Test ({ticker})</h4>"))
        display(bench_res.round({'Hit Rate (%)': 2, 'RMSE': 5, 'MAE': 5}))

def on_change(change):
    render_dashboard(ticker_dropdown.value, threshold_slider.value)

ticker_dropdown.observe(on_change, names='value')
threshold_slider.observe(on_change, names='value')

display(widgets.HBox([ticker_dropdown, threshold_slider]))
display(out)

# Render lan dau
render_dashboard(ticker_dropdown.value, threshold_slider.value)


Output()

## 4. Ket luan & Huong phat trien
- Giao dien demo nay giup truc quan hoa truc tiep cac tin hieu ma mo hinh Deep Learning dua ra tren tung co phieu Top 10.
- Duong Backtest loi nhuan tich luy (Cumulative Return) cho thay loi the phong ngua rui ro cua AI khi thi truong suy giam (tin hieu SELL giup tranh cac nhip sut giam lon cua co phieu).
- Day la ban dung thu (Mockup/Proof of Concept) truoc khi nhom chuyen giao cho Member 3 viet REST API (FastAPI) va Member 4 xay dung ung dung Web Dashboard (Streamlit).